# 第二章 处理文本数据

## 2.2 文本分词
分词工具：https://tiktokenizer.vercel.app/?model=gpt2

In [36]:
import os

import urllib.request  

url ='https://raw.githubusercontent.com/xuweicn1/GPT-From-Zero/refs/heads/ch02/ch02/the-verdict.txt'
file_path = 'the-verdict.txt'

if not os.path.exists('the-verdict.txt'):
    urllib.request.urlretrieve(url, file_path)

with open(file_path, 'r') as f:
    raw_text = f.read()


len(raw_text)

目标： 将20479 个单词分割成独立的单词和特殊字符，方便之后将他们转换为嵌入向量

In [37]:
import re
text = 'Hello, world. This, is a test.'
result = re.split(r'(\s)',text)

In [38]:
result

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']

修正算法

In [39]:
text = 'Hello, world. This, is a test.'
result = re.split(r'([,.]|\s)',text)

In [40]:
result

['Hello',
 ',',
 '',
 ' ',
 'world',
 '.',
 '',
 ' ',
 'This',
 ',',
 '',
 ' ',
 'is',
 ' ',
 'a',
 ' ',
 'test',
 '.',
 '']

In [41]:
text = 'Hello, world. Is This-- is a test?'
result = re.split(r'([,.:;?_!"()\']|--|\s)',text)

In [42]:
result

['Hello',
 ',',
 '',
 ' ',
 'world',
 '.',
 '',
 ' ',
 'Is',
 ' ',
 'This',
 '--',
 '',
 ' ',
 'is',
 ' ',
 'a',
 ' ',
 'test',
 '?',
 '']

In [43]:
result = [item.strip() for item in result if item.strip()]

In [44]:
result

['Hello', ',', 'world', '.', 'Is', 'This', '--', 'is', 'a', 'test', '?']

In [45]:
prosessed = re.split(r'([,.:;?_!"()\']|--|\s)',raw_text)

In [46]:
prosessed = [item.strip() for item in prosessed if item.strip()]

In [47]:
len(prosessed)

4690

In [48]:
prosessed[:30]

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius',
 '--',
 'though',
 'a',
 'good',
 'fellow',
 'enough',
 '--',
 'so',
 'it',
 'was',
 'no',
 'great',
 'surprise',
 'to',
 'me',
 'to',
 'hear',
 'that',
 ',',
 'in']

## 2.3 将词元转换为词元ID

In [49]:
all_words = sorted(set(prosessed))
vocab_size = len(all_words)

In [50]:
vocab_size

1130

创建词汇表

In [51]:
vocab = {token:inter for inter,token in enumerate(all_words)}

实现简单的文本分词器

In [52]:
class SimpleTokenizerV1:
    def __init__(self, vocab=None):
        """
        vocab: 字符串到整数的映射，例如 {'A': 11, 'Ah': 12, 'Among': 13}

        """
        self.str_to_init = vocab
        # 反转字典，得到 init_to_str 示例： {11: 'A', 12: 'Ah', 13: 'Among'}
        self.init_to_str = {v: k for k, v in vocab.items()}

    def encode(self, text):
        """
        将输入文本转换为整数列表
        """
        prosessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        prosessed = [item.strip() for item in prosessed if item.strip()]
        ids = [self.str_to_init[s] for s in prosessed]
        return ids

    def decode(self, ids):
        """
        将整数列表转换回文本
        """

        text = " ".join([self.init_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r"\1", text)

        return text



In [53]:
tokenizer = SimpleTokenizerV1(vocab)

In [54]:
text = """"It's the last he painted,you know,"
Mrs. Gisburn said with pardonable pride. """

In [55]:
ids= tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [56]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


## 2.4 特殊上下文词元处理

In [57]:
text = "Hello, do you like tea"
print(tokenizer.encode(text))

KeyError: 'Hello'

报错原因是我们的词汇表中并没有Hello这个词

In [58]:
all_tokens =sorted(list(set(prosessed)))
all_tokens.extend(["<|endoftext|>","<|unk|>"])
vocab = {token:inter for inter,token in enumerate(all_tokens)}

In [59]:
len(vocab)

1132

In [60]:
for i,item  in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


优化代码，使他能够处理位置单词的文本分词器

In [61]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [62]:
tokenizer = SimpleTokenizerV2(vocab)

In [63]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975]

In [64]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea'

In [65]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [69]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [70]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'